# CosmoML 07 — ML Surrogate vs Theory MCMC

Compares **XGBoost ML surrogate** inference against exact **theoretical χ² MCMC** for w₀wₐCDM cosmological parameters.

**Scenarios**
- Pair 1a · Pantheon+ + DESI BAO
- Pair 1b · Pantheon+ + DESI BAO + Planck CMB prior
- BAO-only · DESI BAO

**Methods compared**
1. ML surrogate — XGBoost GPU booster, 1 024 RWMH chains
2. Theory CPU — exact χ², parallel `ProcessPoolExecutor`
3. Theory GPU — exact χ², JAX `vmap` + JIT (if available)

**Output**: `outputs/paper/{datasets,models,figures,chains}/`


In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

import sys, os, json, time, concurrent.futures, multiprocessing
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import getdist
import getdist.plots
from iminuit import Minuit

ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

from cosmoml.data import load_pantheon_plus, load_desi_bao
from cosmoml.theory import chi2_bao, chi2_joint
from cosmoml.priors import planck_prior_chi2
from cosmoml.sampling import build_chi2_dataset, load_or_build, _chi2_worker
from cosmoml.ml import (
    train_xgb, plot_learning_curve,
    shap_summary, shap_waterfall, shap_dependence_all,
    use_paper_style,
)
from cosmoml.ml.train import LogChi2Model
from cosmoml.ml.marginal import _parallel_mcmc, _render_getdist, _sokal_tau

try:
    from cosmoml.theory.jax_theory import make_chi2_gpu_fn as _make_chi2_gpu_fn
    _JAX_OK = True
    print('JAX GPU theory: AVAILABLE')
except ImportError:
    _JAX_OK = False
    print('JAX GPU theory: NOT AVAILABLE (pip install jax[cuda])')

use_paper_style()
print('Setup complete.')


In [ ]:
# Paths — compartidos con run_cosmo.py (mismos nombres de sección)
PAPER_DIR    = Path('..').resolve() / 'outputs' / 'paper'
DATASETS_DIR = PAPER_DIR / 'datasets'
MODELS_DIR   = PAPER_DIR / 'models'
FIGURES_DIR  = PAPER_DIR / 'figures'
CHAINS_DIR   = PAPER_DIR / 'chains'
for _d in (DATASETS_DIR, MODELS_DIR, FIGURES_DIR, CHAINS_DIR):
    _d.mkdir(parents=True, exist_ok=True)

FORCE_RETRAIN = False  # True → regenera datasets, modelos y cadenas

GLOBAL_RANGES = {
    'Om': (0.1,   0.9),
    'H0': (20.0, 100.0),
    'w0': (-3.0,  0.2),
    'wa': (-3.0,  2.0),
}
LABELS = {
    'Om': r'$\Omega_m$',
    'H0': r'$H_0$',
    'w0': r'$w_0$',
    'wa': r'$w_a$',
}
MARKERS_DE = {'w0': -1.0, 'wa': 0.0}
FEATURES_4 = ['Om', 'H0', 'w0', 'wa']
RANGES_4   = GLOBAL_RANGES.copy()

print(f'PAPER_DIR     : {PAPER_DIR}')
print(f'FORCE_RETRAIN : {FORCE_RETRAIN}')


In [ ]:
print('Cargando datos observacionales...')
panth = load_pantheon_plus(apply_mask=True)
bao   = load_desi_bao()
print(f'  Pantheon+ : {len(panth)} SNe')
print(f'  DESI BAO  : {len(bao)} medidas')


In [ ]:
# ── Chi2 functions — module-level para pickle con ProcessPoolExecutor ─────────

def chi2_panth_bao(Om, H0, w0, wa):
    """w0waCDM: Pantheon+ + DESI BAO (sin calibradores Cepheid)."""
    return chi2_joint(panth, bao, Om=Om, H0=H0, w0=w0, wa=wa,
                      sne_kwargs={'use_cepheid_calibrators': False})

def chi2_panth_bao_cmb(Om, H0, w0, wa):
    """w0waCDM: Pantheon+ + DESI BAO + prior Planck CMB."""
    return chi2_panth_bao(Om, H0, w0, wa) + planck_prior_chi2(Om=Om, H0=H0)

def chi2_bao_only(Om, H0, w0, wa):
    """w0waCDM: DESI BAO solo."""
    return chi2_bao(bao, Om=Om, H0=H0, w0=w0, wa=wa)

print('Chi2 functions definidas.')


In [ ]:
def _make_gpu_predict_fn(model):
    """Booster GPU para MCMC rapido (evita copia CPU en cada step)."""
    if isinstance(model, LogChi2Model):
        booster = model.raw_model.get_booster().copy()
        booster.set_param({'device': 'cuda'})
        y_min = model.y_min
        def predict_fn(arr):
            log_y = booster.inplace_predict(arr.astype(np.float32), iteration_range=(0, 0))
            return (10.0 ** log_y) - 1.0 + y_min
    else:
        booster = model.get_booster().copy()
        booster.set_param({'device': 'cuda'})
        def predict_fn(arr):
            return booster.inplace_predict(arr.astype(np.float32), iteration_range=(0, 0))
    return predict_fn


In [ ]:
def _make_theory_predict_fn(chi2_fn, features, executor, n_cores):
    """Predict function CPU paralela para _parallel_mcmc."""
    chunksize = max(1, 1024 // (n_cores * 4))
    def predict_fn(arr):
        tasks = [
            (chi2_fn, {f: float(arr[i, j]) for j, f in enumerate(features)})
            for i in range(len(arr))
        ]
        return np.array(list(executor.map(_chi2_worker, tasks, chunksize=chunksize)), dtype=float)
    return predict_fn


In [ ]:
def locate_bestfit(chi2_fn, features, ranges):
    init = {f: (ranges[f][0] + ranges[f][1]) / 2.0 for f in features}
    if 'w0' in features: init['w0'] = -1.0
    if 'wa' in features: init['wa'] = 0.0
    if 'Om' in features: init['Om'] = 0.3
    if 'H0' in features: init['H0'] = 68.0
    m = Minuit(chi2_fn, **init)
    m.limits = [ranges[f] for f in features]
    m.migrad()
    REF = {f: float(m.values[f]) for f in features}
    print(f"  Best-fit : {', '.join(f'{f}={v:.4f}' for f, v in REF.items())}")
    print(f'  chi2_min : {m.fval:.2f}')
    return REF, m.fval


def build_dataset(chi2_fn, section, features, ranges):
    csv_path = DATASETS_DIR / f'{section}_dataset.csv'
    REF, _ = locate_bestfit(chi2_fn, features, ranges)
    ndim = len(features)

    def builder():
        slices = []
        for _i in range(ndim):
            for _j in range(_i + 1, ndim):
                fi, fj = features[_i], features[_j]
                fixed = {f: REF[f] for f in features if f not in (fi, fj)}
                slices.append({fi: ranges[fi], fj: ranges[fj], **fixed, '_n': 20_000})
        return build_chi2_dataset(
            chi2_fn=chi2_fn,
            param_names=features,
            slices=slices,
            random_box={f: ranges[f] for f in features},
            n_random=80_000,
            save_to=csv_path,
            seed=42,
        )

    df = load_or_build(csv_path, builder, force=FORCE_RETRAIN)
    print(f'  Dataset  : {len(df):,} rows | chi2 [{df["chi2"].min():.2f}, {df["chi2"].max():.2f}]')
    return df, REF


In [ ]:
def train_and_shap(df, features, section, title=''):
    model, info = train_xgb(
        df, features=features,
        log_target=True,
        hp_overrides=dict(n_estimators=5000, learning_rate=0.03, max_depth=10, device='cuda'),
        cache_path=MODELS_DIR / f'{section}_model.ubj',
        force_retrain=FORCE_RETRAIN,
    )
    plot_learning_curve(
        info,
        title=f'{title} — Learning Curve (R²={info["r2"]:.5f})',
        save_path=FIGURES_DIR / f'{section}_learning_curve.png',
        show=True,
    )
    shap_v, X_s = shap_summary(
        model, info['X_val'],
        title=f'{title} — SHAP',
        save_dir=FIGURES_DIR,
        prefix=section,
        show=True,
    )
    shap_waterfall(
        shap_v, idx=0,
        title=f'{title} — SHAP waterfall',
        save_path=FIGURES_DIR / f'{section}_shap_waterfall.png',
        show=True,
    )
    return model, info, shap_v, X_s


In [ ]:
def run_mcmc_ml(model, features, ranges, ref, section, labels, markers=None, title=''):
    chain_path  = CHAINS_DIR  / f'{section}_samples.npy'
    figure_path = FIGURES_DIR / f'{section}_getdist.png'
    str_labels  = [labels.get(f, f).replace('$', '') for f in features]
    lows   = np.array([ranges[f][0] for f in features])
    highs  = np.array([ranges[f][1] for f in features])
    center = np.array([ref[f]       for f in features])

    if chain_path.exists() and not FORCE_RETRAIN:
        samples = np.load(chain_path)
        print(f'  [ML]  Cache: {chain_path}  ({len(samples):,} samples)')
        meta = {'cached': True, 'wall_s': None, 'n_samples': len(samples)}
    else:
        print('  [ML]  Iniciando MCMC GPU (1024 chains)...')
        predict_fn = _make_gpu_predict_fn(model)
        t0 = time.perf_counter()
        samples = _parallel_mcmc(
            predict_fn, lows, highs, center, len(features),
            n_chains=1024, n_steps=10_000, burn_in=500, seed=42, ess_target=10_000,
        )
        wall = time.perf_counter() - t0
        np.save(chain_path, samples)
        print(f'  [ML]  Guardado: {chain_path}  ({wall:.1f}s, {len(samples):,} muestras)')
        meta = {'cached': False, 'wall_s': round(wall, 2), 'n_samples': len(samples)}

    fig = _render_getdist(samples, features, str_labels, markers, title,
                          smooth_scale=0.5, ranges=ranges)
    fig.savefig(figure_path, dpi=150, bbox_inches='tight')
    plt.show()
    return samples, meta


In [ ]:
def run_mcmc_theory_cpu(chi2_fn, features, ranges, ref, section,
                         labels, markers=None, title=''):
    chain_path  = CHAINS_DIR  / f'{section}_samples_theory.npy'
    figure_path = FIGURES_DIR / f'{section}_getdist_theory.png'
    str_labels  = [labels.get(f, f).replace('$', '') for f in features]
    n_cores = max(1, multiprocessing.cpu_count() - 1)
    lows   = np.array([ranges[f][0] for f in features])
    highs  = np.array([ranges[f][1] for f in features])
    center = np.array([ref[f]       for f in features])

    if chain_path.exists() and not FORCE_RETRAIN:
        samples = np.load(chain_path)
        print(f'  [TH]  Cache: {chain_path}  ({len(samples):,} samples)')
        meta = {'cached': True, 'wall_s': None, 'n_samples': len(samples)}
    else:
        print(f'  [TH]  MCMC teórico CPU ({n_cores} cores)...')
        t0 = time.perf_counter()
        with concurrent.futures.ProcessPoolExecutor(max_workers=n_cores) as executor:
            predict_fn = _make_theory_predict_fn(chi2_fn, features, executor, n_cores)
            samples = _parallel_mcmc(
                predict_fn, lows, highs, center, len(features),
                n_chains=1024, n_steps=10_000, burn_in=500, seed=42, ess_target=10_000,
            )
        wall = time.perf_counter() - t0
        np.save(chain_path, samples)
        print(f'  [TH]  Guardado: {chain_path}  ({wall:.1f}s)')
        meta = {'cached': False, 'wall_s': round(wall, 2), 'n_samples': len(samples)}

    fig = _render_getdist(samples, features, str_labels, markers, title + ' [Theory CPU]',
                          smooth_scale=0.5, ranges=ranges)
    fig.savefig(figure_path, dpi=150, bbox_inches='tight')
    plt.show()
    return samples, meta


def run_mcmc_theory_gpu(gpu_predict_fn, features, ranges, ref, section,
                         labels, markers=None, title=''):
    chain_path  = CHAINS_DIR  / f'{section}_samples_theory_gpu.npy'
    figure_path = FIGURES_DIR / f'{section}_getdist_theory_gpu.png'
    str_labels  = [labels.get(f, f).replace('$', '') for f in features]
    lows   = np.array([ranges[f][0] for f in features])
    highs  = np.array([ranges[f][1] for f in features])
    center = np.array([ref[f]       for f in features])

    if chain_path.exists() and not FORCE_RETRAIN:
        samples = np.load(chain_path)
        print(f'  [TH-GPU]  Cache: {chain_path}  ({len(samples):,} samples)')
        meta = {'cached': True, 'wall_s': None, 'n_samples': len(samples)}
    else:
        print('  [TH-GPU]  MCMC teórico JAX/GPU...')
        t0 = time.perf_counter()
        samples = _parallel_mcmc(
            gpu_predict_fn, lows, highs, center, len(features),
            n_chains=1024, n_steps=10_000, burn_in=500, seed=42, ess_target=10_000,
        )
        wall = time.perf_counter() - t0
        np.save(chain_path, samples)
        print(f'  [TH-GPU]  Guardado: {chain_path}  ({wall:.1f}s)')
        meta = {'cached': False, 'wall_s': round(wall, 2), 'n_samples': len(samples)}

    fig = _render_getdist(samples, features, str_labels, markers, title + ' [Theory GPU]',
                          smooth_scale=0.5, ranges=ranges)
    fig.savefig(figure_path, dpi=150, bbox_inches='tight')
    plt.show()
    return samples, meta


In [ ]:
def plot_comparison(samples_list, dataset_labels, features, labels,
                     markers=None, title='', save_path=None,
                     filled=None, ranges=None):
    COLORS = ['#0044cc', '#cc0000', '#009933', '#cc6600']
    n = len(samples_list)
    if filled is None:
        filled = [True] + [False] * (n - 1)
    str_labels = [labels.get(f, f).replace('$', '') for f in features]
    mc_ranges  = {f: list(r) for f, r in ranges.items()} if ranges else None

    mc_list = [
        getdist.MCSamples(
            samples=s, names=features, labels=str_labels, label=dl,
            ranges=mc_ranges,
            settings={'smooth_scale_2D': 0.5, 'smooth_scale_1D': 0.5},
        )
        for s, dl in zip(samples_list, dataset_labels)
    ]
    g = getdist.plots.get_subplot_plotter()
    g.triangle_plot(
        mc_list, filled=filled,
        contour_colors=COLORS[:n], contour_lws=[2.0] * n,
        markers=markers,
        marker_args={'ls': '--', 'color': 'gray', 'lw': 1.5, 'alpha': 0.8} if markers else None,
        legend_labels=dataset_labels,
        legend_loc='upper right',
    )
    if ranges:
        for _i in range(len(features)):
            for _j in range(_i + 1):
                _ax = g.subplots[_i][_j]
                if _ax is None: continue
                _ax.set_xlim(*ranges[features[_j]])
                if _i != _j: _ax.set_ylim(*ranges[features[_i]])
    if title:
        g.fig.suptitle(title, fontsize=13, y=1.01)
    if save_path:
        g.fig.savefig(save_path, dpi=200, bbox_inches='tight')
        print(f'  Guardado: {save_path}')
    plt.show()
    return g.fig


def _gen_param_array(features, ranges, ref):
    """Replica el muestreo de build_dataset (seed=42) para comparacion GPU vs CPU."""
    ndim = len(features)
    rng  = np.random.default_rng(42)
    slices_spec = []
    for _i in range(ndim):
        for _j in range(_i + 1, ndim):
            fi, fj = features[_i], features[_j]
            fixed  = {f: ref[f] for f in features if f not in (fi, fj)}
            slices_spec.append({fi: ranges[fi], fj: ranges[fj], **fixed})
    n_slice, n_random = 20_000, 80_000
    blocks = []
    for spec in slices_spec:
        block = {k: (rng.uniform(v[0], v[1], n_slice) if isinstance(v, tuple)
                     else np.full(n_slice, float(v)))
                 for k, v in spec.items()}
        blocks.append(block)
    random_block = {f: rng.uniform(ranges[f][0], ranges[f][1], n_random) for f in features}
    blocks.append(random_block)
    params = {f: np.concatenate([b[f] for b in blocks]) for f in features}
    arr = np.column_stack([params[f] for f in features]).astype(np.float32)
    return arr, params


def build_dataset_gpu(gpu_chi2_fn, chi2_fn_cpu, section, features, ranges, ref):
    csv_gpu = DATASETS_DIR / f'{section}_dataset_gpu.csv'
    csv_cpu = DATASETS_DIR / f'{section}_dataset.csv'
    n_cores = max(1, multiprocessing.cpu_count() - 1)
    arr, params = _gen_param_array(features, ranges, ref)
    total = len(arr)

    if csv_gpu.exists() and not FORCE_RETRAIN:
        print(f'  [DS-GPU]  Cache: {csv_gpu} ({total:,} pts)')
        df_gpu = pd.read_csv(csv_gpu)
        gpu_wall = None
    else:
        print(f'  [DS-GPU]  Evaluando {total:,} puntos (JAX GPU batch)...')
        t0 = time.perf_counter()
        chi2_gpu = gpu_chi2_fn(arr)
        gpu_wall = time.perf_counter() - t0
        print(f'  [DS-GPU]  {gpu_wall:.2f}s  ({total / gpu_wall:,.0f} pts/s)')
        df_gpu = pd.DataFrame({**params, 'chi2': chi2_gpu})[features + ['chi2']]
        df_gpu.to_csv(csv_gpu, index=False)

    N_CPU = 5_000
    sample_arr = arr[:N_CPU]
    tasks = [(chi2_fn_cpu, {f: float(sample_arr[i, j]) for j, f in enumerate(features)})
             for i in range(N_CPU)]
    print(f'  [DS-CPU]  Timing {N_CPU:,} pts ({n_cores} cores)...')
    t0 = time.perf_counter()
    with concurrent.futures.ProcessPoolExecutor(max_workers=n_cores) as ex:
        list(ex.map(_chi2_worker, tasks, chunksize=100))
    cpu_sample_wall = time.perf_counter() - t0
    cpu_extrap = cpu_sample_wall / N_CPU * total
    speedup = (cpu_extrap / gpu_wall) if (gpu_wall and gpu_wall > 0) else None
    print(f'  [DS-CPU]  {cpu_sample_wall:.1f}s → extrap full: {cpu_extrap:.0f}s')
    if speedup:
        print(f'  [DS-SPD]  Speedup GPU vs CPU: {speedup:.1f}x')

    cmp = {}
    if csv_cpu.exists():
        df_cpu = pd.read_csv(csv_cpu)
        n_cmp = min(len(df_cpu), len(df_gpu))
        diff = np.abs(df_cpu['chi2'].values[:n_cmp] - df_gpu['chi2'].values[:n_cmp])
        cmp  = {'max_abs': float(diff.max()), 'mean_abs': float(diff.mean())}
        print(f'  [DS-CMP]  max|Dchi2|={cmp["max_abs"]:.4f} | mean={cmp["mean_abs"]:.4f}')

    return df_gpu, {
        'gpu_wall_s': round(gpu_wall, 2) if gpu_wall else None,
        'cpu_sample_s': round(cpu_sample_wall, 2),
        'cpu_extrap_s': round(cpu_extrap, 1),
        'speedup': round(speedup, 1) if speedup else None,
        'n_pts': int(total), 'cmp': cmp,
    }


---
# Pair 1 — w₀wₐCDM: Pantheon+ + DESI BAO

Pair 1a (sin prior CMB) y 1b (con prior Planck).  
Necesarios para el resumen 3-way de la Section 3.


In [ ]:
SECTION_2_1A = '6_2_1a'
TITLE_2_1A   = 'w0waCDM - Pantheon+ + DESI BAO'
print(f'=== {TITLE_2_1A} ===')

df_2_1a, REF_2_1a = build_dataset(chi2_panth_bao, SECTION_2_1A, FEATURES_4, RANGES_4)
model_2_1a, info_2_1a, shap_v_2_1a, X_s_2_1a = train_and_shap(
    df_2_1a, FEATURES_4, SECTION_2_1A, title=TITLE_2_1A
)
samples_2_1a, ml_meta_2_1a = run_mcmc_ml(
    model_2_1a, FEATURES_4, RANGES_4, REF_2_1a, SECTION_2_1A,
    labels=LABELS, markers=MARKERS_DE, title=TITLE_2_1A,
)


In [ ]:
SECTION_2_1B = '6_2_1b'
TITLE_2_1B   = 'w0waCDM - Pantheon+ + DESI BAO + CMB'
print(f'=== {TITLE_2_1B} ===')

df_2_1b, REF_2_1b = build_dataset(chi2_panth_bao_cmb, SECTION_2_1B, FEATURES_4, RANGES_4)
model_2_1b, info_2_1b, shap_v_2_1b, X_s_2_1b = train_and_shap(
    df_2_1b, FEATURES_4, SECTION_2_1B, title=TITLE_2_1B
)
samples_2_1b, ml_meta_2_1b = run_mcmc_ml(
    model_2_1b, FEATURES_4, RANGES_4, REF_2_1b, SECTION_2_1B,
    labels=LABELS, markers=MARKERS_DE, title=TITLE_2_1B,
)


---
# BAO-only — w₀wₐCDM: DESI BAO

Pipeline completo: dataset → XGBoost (GPU) → SHAP → ML MCMC.


In [ ]:
SECTION_3_BAO = '6_3_bao'
TITLE_3_BAO   = 'w0waCDM - DESI BAO only'
print(f'=== {TITLE_3_BAO} === (dataset + train)')

df_3_bao, REF_3_bao = build_dataset(chi2_bao_only, SECTION_3_BAO, FEATURES_4, RANGES_4)
model_3_bao, info_3_bao, shap_v_3_bao, X_s_3_bao = train_and_shap(
    df_3_bao, FEATURES_4, SECTION_3_BAO, title=TITLE_3_BAO
)


In [ ]:
print(f'=== {TITLE_3_BAO} === (ML MCMC)')
samples_3_bao, ml_meta_bao = run_mcmc_ml(
    model_3_bao, FEATURES_4, RANGES_4, REF_3_bao, SECTION_3_BAO,
    labels=LABELS, markers=MARKERS_DE, title=TITLE_3_BAO,
)


In [ ]:
print('=== 3-way ML: BAO / BAO+Panth / BAO+Panth+CMB ===')
plot_comparison(
    [samples_3_bao, samples_2_1a, samples_2_1b],
    ['BAO only', 'BAO + Pantheon+', 'BAO + Pantheon+ + CMB'],
    FEATURES_4, LABELS,
    markers=MARKERS_DE,
    title='w0waCDM - ML: BAO / BAO+Pantheon+ / BAO+Pantheon++CMB',
    save_path=FIGURES_DIR / '07_3way_ml.png',
    filled=[True, True, True],
    ranges=RANGES_4,
)


---
# Theory MCMC — CPU Benchmark

χ² exacta evaluada en paralelo con `ProcessPoolExecutor` en todos los cores CPU.  
Mismo sampler `_parallel_mcmc` (1 024 cadenas RWMH) que el ML.


In [ ]:
print(f'=== Theory CPU: {TITLE_3_BAO} ===')
samples_th_bao, th_meta_bao = run_mcmc_theory_cpu(
    chi2_bao_only, FEATURES_4, RANGES_4, REF_3_bao, SECTION_3_BAO,
    labels=LABELS, markers=MARKERS_DE, title=TITLE_3_BAO,
)


In [ ]:
print(f'=== Theory CPU: {TITLE_2_1A} ===')
samples_th_pb, th_meta_pb = run_mcmc_theory_cpu(
    chi2_panth_bao, FEATURES_4, RANGES_4, REF_2_1a, SECTION_2_1A,
    labels=LABELS, markers=MARKERS_DE, title=TITLE_2_1A,
)

print(f'=== Theory CPU: {TITLE_2_1B} ===')
samples_th_pbc, th_meta_pbc = run_mcmc_theory_cpu(
    chi2_panth_bao_cmb, FEATURES_4, RANGES_4, REF_2_1b, SECTION_2_1B,
    labels=LABELS, markers=MARKERS_DE, title=TITLE_2_1B,
)


In [ ]:
print('=== 3-way Theory CPU ===')
plot_comparison(
    [samples_th_bao, samples_th_pb, samples_th_pbc],
    ['BAO [Theory]', 'BAO+Panth [Theory]', 'BAO+Panth+CMB [Theory]'],
    FEATURES_4, LABELS,
    markers=MARKERS_DE,
    title='w0waCDM - Theory CPU: BAO / BAO+Panth / BAO+Panth+CMB',
    save_path=FIGURES_DIR / '07_3way_theory_cpu.png',
    filled=[True, True, True],
    ranges=RANGES_4,
)


---
# Theory MCMC — GPU Benchmark (JAX)

χ² exacta vectorizada con JAX `vmap` + JIT.  
Una sola llamada batched por step de MCMC sobre la GPU → máximo throughput.


In [ ]:
samples_tg_bao = samples_tg_pb = samples_tg_pbc = None
th_gpu_meta_bao = th_gpu_meta_pb = th_gpu_meta_pbc = None

if _JAX_OK:
    print('Compilando funciones JAX (warm-up)...')
    gpu_fn_bao = _make_chi2_gpu_fn(panth=None,  bao=bao)
    gpu_fn_pb  = _make_chi2_gpu_fn(panth=panth, bao=bao)
    gpu_fn_pbc = _make_chi2_gpu_fn(panth=panth, bao=bao, planck_prior=True)
    print('JAX JIT listo.\n')

    print(f'=== Theory GPU: {TITLE_3_BAO} ===')
    samples_tg_bao, th_gpu_meta_bao = run_mcmc_theory_gpu(
        gpu_fn_bao, FEATURES_4, RANGES_4, REF_3_bao, SECTION_3_BAO,
        labels=LABELS, markers=MARKERS_DE, title=TITLE_3_BAO,
    )

    print(f'=== Theory GPU: {TITLE_2_1A} ===')
    samples_tg_pb, th_gpu_meta_pb = run_mcmc_theory_gpu(
        gpu_fn_pb, FEATURES_4, RANGES_4, REF_2_1a, SECTION_2_1A,
        labels=LABELS, markers=MARKERS_DE, title=TITLE_2_1A,
    )

    print(f'=== Theory GPU: {TITLE_2_1B} ===')
    samples_tg_pbc, th_gpu_meta_pbc = run_mcmc_theory_gpu(
        gpu_fn_pbc, FEATURES_4, RANGES_4, REF_2_1b, SECTION_2_1B,
        labels=LABELS, markers=MARKERS_DE, title=TITLE_2_1B,
    )
else:
    print('JAX no disponible — Theory GPU MCMC omitido.')
    print("Instala con: pip install 'jax[cuda]'")


In [ ]:
if all(s is not None for s in [samples_tg_bao, samples_tg_pb, samples_tg_pbc]):
    print('=== 3-way Theory GPU ===')
    plot_comparison(
        [samples_tg_bao, samples_tg_pb, samples_tg_pbc],
        ['BAO [Theory GPU]', 'BAO+Panth [Theory GPU]', 'BAO+Panth+CMB [Theory GPU]'],
        FEATURES_4, LABELS,
        markers=MARKERS_DE,
        title='w0waCDM - Theory GPU: BAO / BAO+Panth / BAO+Panth+CMB',
        save_path=FIGURES_DIR / '07_3way_theory_gpu.png',
        filled=[True, True, True],
        ranges=RANGES_4,
    )
else:
    print('(JAX no disponible — plot omitido)')


---
# ML vs Theory — Comparación directa

Overlay del surrogate ML con la teoría exacta (CPU y GPU si disponible).  
Verifica que el posterior del surrogate coincide con el posterior teórico.


In [ ]:
def ml_vs_theory(ml_s, th_cpu_s, th_gpu_s, features, ranges, labels, markers,
                   title, save_path):
    slist   = [ml_s, th_cpu_s]
    dlabels = ['ML Surrogate', 'Theory CPU']
    filled  = [True, False]
    if th_gpu_s is not None:
        slist.append(th_gpu_s)
        dlabels.append('Theory GPU')
        filled.append(False)
    plot_comparison(slist, dlabels, features, labels,
                    markers=markers, title=title, save_path=save_path,
                    filled=filled, ranges=ranges)


print('=== ML vs Theory: BAO-only ===')
ml_vs_theory(
    samples_3_bao, samples_th_bao, samples_tg_bao,
    FEATURES_4, RANGES_4, LABELS, MARKERS_DE,
    title='w0waCDM - BAO only: ML vs Theory',
    save_path=FIGURES_DIR / '07_ml_vs_theory_bao.png',
)


In [ ]:
print('=== ML vs Theory: Pantheon+ + BAO ===')
ml_vs_theory(
    samples_2_1a, samples_th_pb, samples_tg_pb,
    FEATURES_4, RANGES_4, LABELS, MARKERS_DE,
    title='w0waCDM - Pantheon+ + BAO: ML vs Theory',
    save_path=FIGURES_DIR / '07_ml_vs_theory_panth_bao.png',
)

print('=== ML vs Theory: Pantheon+ + BAO + CMB ===')
ml_vs_theory(
    samples_2_1b, samples_th_pbc, samples_tg_pbc,
    FEATURES_4, RANGES_4, LABELS, MARKERS_DE,
    title='w0waCDM - Pantheon+ + BAO + CMB: ML vs Theory',
    save_path=FIGURES_DIR / '07_ml_vs_theory_panth_bao_cmb.png',
)


---
# Dataset Generation: GPU vs CPU

Compara JAX GPU batch vs CPU paralelo para generar el dataset de entrenamiento.  
Misma semilla (seed=42) y mismo muestreo que `build_dataset`.


In [ ]:
ds_meta_bao = ds_meta_pb = ds_meta_pbc = None

if _JAX_OK:
    print('=== Dataset GPU: BAO-only ===')
    _, ds_meta_bao = build_dataset_gpu(
        gpu_fn_bao, chi2_bao_only, SECTION_3_BAO, FEATURES_4, RANGES_4, REF_3_bao
    )
    print('=== Dataset GPU: Pantheon+ + BAO ===')
    _, ds_meta_pb = build_dataset_gpu(
        gpu_fn_pb, chi2_panth_bao, SECTION_2_1A, FEATURES_4, RANGES_4, REF_2_1a
    )
    print('=== Dataset GPU: Pantheon+ + BAO + CMB ===')
    _, ds_meta_pbc = build_dataset_gpu(
        gpu_fn_pbc, chi2_panth_bao_cmb, SECTION_2_1B, FEATURES_4, RANGES_4, REF_2_1b
    )

    print('\n' + '-'*82)
    print(f'{"scenario":<22}{"gpu [s]":>10}{"cpu extrap":>12}{"speedup":>10}{"max|Dchi2|":>12}')
    print('-'*82)
    for name, m in [(SECTION_3_BAO, ds_meta_bao), (SECTION_2_1A, ds_meta_pb), (SECTION_2_1B, ds_meta_pbc)]:
        if m is None: continue
        gpu_s = f"{m['gpu_wall_s']:.1f}s" if m['gpu_wall_s'] else 'cached'
        cpu_s = f"{m['cpu_extrap_s']:.0f}s"
        spd   = f"{m['speedup']:.1f}x" if m['speedup'] else '-'
        maxd  = f"{m['cmp']['max_abs']:.4f}" if m.get('cmp') else '-'
        print(f'{name:<22}{gpu_s:>10}{cpu_s:>12}{spd:>10}{maxd:>12}')
    print('-'*82)
else:
    print('JAX no disponible — Dataset GPU omitido.')


---
# Timing Summary

Tiempos de pared para los tres métodos de inferencia en cada escenario.


In [ ]:
_N  = lambda m: m if m is not None else {'cached': True, 'wall_s': None, 'n_samples': None}
_fs = lambda v: f'{v:.1f}s' if v else 'cached'

def _spd(ml_m, th_m):
    w_ml, w_th = ml_m.get('wall_s'), th_m.get('wall_s')
    return round(w_th / w_ml, 2) if (w_ml and w_th) else None

_fsp = lambda v: f'{v:.1f}x' if v else '-'

rows = [
    ('bao_only',      ml_meta_bao,   th_meta_bao, _N(th_gpu_meta_bao)),
    ('panth_bao',     ml_meta_2_1a,  th_meta_pb,  _N(th_gpu_meta_pb)),
    ('panth_bao_cmb', ml_meta_2_1b,  th_meta_pbc, _N(th_gpu_meta_pbc)),
]

print('-'*90)
print(f'{"scenario":<18}{"ml [s]":>10}{"th-cpu [s]":>12}{"th-gpu [s]":>12}'
      f'{"x cpu/ml":>10}{"x gpu/ml":>10}{"x gpu/cpu":>10}')
print('-'*90)
for name, ml_m, th_m, tg_m in rows:
    print(
        f'{name:<18}'
        f'{_fs(ml_m["wall_s"]):>10}'
        f'{_fs(th_m["wall_s"]):>12}'
        f'{_fs(tg_m["wall_s"]):>12}'
        f'{_fsp(_spd(ml_m, th_m)):>10}'
        f'{_fsp(_spd(ml_m, tg_m)):>10}'
        f'{_fsp(_spd(tg_m, th_m)):>10}'
    )
print('-'*90)

timings = {
    'engines': {
        'ml':         'RWMH 1024 chains, GPU XGBoost booster (LogChi2)',
        'theory_cpu': 'RWMH 1024 chains, CPU ProcessPoolExecutor (exact chi2)',
        'theory_gpu': 'RWMH 1024 chains, JAX vmap GPU (exact chi2)',
    },
    'common': {'n_chains': 1024, 'n_steps_cap': 10_000, 'burn_in': 500,
               'seed': 42, 'ess_target': 10_000},
    'runs': {
        name: {
            'ml': ml_m, 'theory_cpu': th_m, 'theory_gpu': tg_m,
            'speedup_cpu_vs_ml':  _spd(ml_m, th_m),
            'speedup_gpu_vs_ml':  _spd(ml_m, tg_m),
            'speedup_gpu_vs_cpu': _spd(tg_m, th_m),
        }
        for name, ml_m, th_m, tg_m in rows
    },
}
timings_path = PAPER_DIR / 'timings_07.json'
with open(timings_path, 'w') as f:
    json.dump(timings, f, indent=2)
print(f'\nTimings guardados: {timings_path}')
print(f'Figuras          : {FIGURES_DIR}/')
print(f'Chains           : {CHAINS_DIR}/')
